In [ ]:
%reload_ext autoreload
%autoreload 2

import os, glob, sys
import pandeia.engine
import stsynphot
import synphot
from synphot import SpectralElement
from synphot.models import Empirical1D
import numpy as np
from astropy.io import fits, ascii
import astropy.units as u
import astropy.constants as const
from astropy.table import QTable, Table, Column
from scipy import interpolate
from scipy.integrate import simpson
import matplotlib.pyplot as plt
import matplotlib as mpl
from rsg_phot.rsg import rsg_phot
from rsg_phot.dust import dusty_gen
from tqdm import tqdm
import copy
import pandas as pd
pd.options.mode.chained_assignment = None
%matplotlib inline

### DUSTY models

In [ ]:
logg = np.array([0, 1, 2, 3])
z = np.array([-2.5, -2.0, -1.50, -1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0])

In [ ]:
taus = np.hstack([np.linspace(0.0, 6.0, 16), np.linspace(7, 15, 9)])
with open('data/r2/dusty_sil_grid/dusty_out/rsg_2600.0_500.0/taugrid.dat', 'w') as f:
    f.write('lambda0 = 0.55 micron\n')

with open('data/r2/dusty_sil_grid/dusty_out/rsg_2600.0_500.0/taugrid.dat', 'ab') as f:
    np.savetxt(f, taus, fmt='%.2f')

In [ ]:
dustgen = dusty_gen(dist = 10 * u.Mpc, outdir='/Users/aswin/rsg_phot/data/dusty_sil_grid')
tb = dustgen.dust_spec([0.01, 6, 21], 4.0, 3500*u.K, 800*u.K, 
                       dust_comp='sil', shell_thickness=2, filedir=None, 
                       redo_input=False, redo_dusty=True, tb_overwrite=True)

In [ ]:
tb

In [ ]:
tb['lambda']

In [ ]:
for cl in tb.colnames[1:]:  
    plt.plot(tb['lambda'], tb[cl])
plt.xscale('log')
plt.yscale('log')

In [ ]:
Z = 0.00
sg = '+' if Z > -1e-5 else '-'
marcs_path = '/Users/aswin/rsg_phot/data/marcs/g0.00/'
mrc = np.loadtxt(os.path.join(marcs_path, 'Z{sg}{met:.2f}'.format(sg=sg, met=Z), 
                              'MARCS_models_g0.00_Z{sg}{met:.2f}.dat'.format(sg=sg, met=Z)))
t = mrc[0]
flx = mrc[1:].T
# flx = Table(fl, names=t, units=[u.erg/u.s/u.cm**2/u.Angstrom]*len(t))

w = np.logspace(np.log10(1300.), np.log10(200000.), num=5036)

In [ ]:
t = np.delete(t, [5], axis=0)
flx = np.delete(flx, [5], axis=0)

In [ ]:
from scipy import interpolate
inp = interpolate.RegularGridInterpolator((w, t), flx.T, method='cubic', bounds_error=False)
wavelength_grid, temp_grid = np.meshgrid(w, [3600], indexing='ij', sparse=True)
nf = inp((wavelength_grid, temp_grid)).flatten()
# nf = 10**nf
nf[nf<0] = np.min(flx)

In [ ]:
plt.plot(w/1e4, flx[5])
plt.plot(w/1e4, nf, alpha=0.5)
plt.xscale('log')
plt.yscale('log')

In [ ]:
# plt.plot(w/1e4, flx[5])
plt.plot(w/1e4, nf)
plt.xlim(1, 2)
plt.ylim(0, 8e5)

In [ ]:
plt.plot(w/1e4, nf/flx[5])

In [ ]:
plt.plot(w, nf)
plt.plot(w, flx[4], alpha=0.3)
plt.plot(w, flx[5], alpha=0.3)
plt.plot(w, flx[6], alpha=0.3)
plt.xlim(7000, 20000)
plt.ylim(10**5, 10**6)
plt.xscale('log')
plt.yscale('log')

In [ ]:
plt.scatter(w/1e4, 100*(nf - flx[5]) / flx[5], s=0.1)
plt.xlim(0.5, 4.5)
plt.ylim(-1, 5)

In [ ]:
mrc = np.loadtxt(os.path.join(marcs_path, 'Z{sg}{met:.2f}'.format(sg=sg, met=Z), 
                              'MARCS_models_g0.00_Z{sg}{met:.2f}.dat'.format(sg=sg, met=Z)),
                              unpack=True)

In [ ]:
mrc[:, 0]

In [ ]:
for i in range(len(t)):
    plt.plot((w/1e4), flx[i], label = t[i])
plt.legend()
plt.xlabel(r'$\lambda(\mu m)$')
plt.xscale('log')
plt.yscale('log')

### MARCS models

In [ ]:
syn_rsg = rsg_phot()
rsgspec = syn_rsg.create_rsg(6.0, 4.4 * u.Lsun, 3500 * u.K, 800 * u.K, 13.0 * u.Mpc, 'g2')
filts = ['f090w', 'f115w', 'f150w', 'f164n', 'f182m', 'f187n', 'f200w', 'f210m', 'f212n', 'f277w', 'f335m', 'f356w', 'f360m', 'f405n', 'f444w']
rsgmag_s2 = syn_rsg.compute_rsg_mag(filts, 15.0, 4.4 * u.Lsun, 3500 * u.K, 1000 * u.K, 13.0 * u.Mpc, 's2')
rsgmag_g2 = syn_rsg.compute_rsg_mag(filts, 15.0, 4.4 * u.Lsun, 3500 * u.K, 1000 * u.K, 13.0 * u.Mpc, 'g2')

In [ ]:
wvs = rsgspec.waveset
rsgspec(wvs)

In [ ]:
wvs = rsgspec.waveset
plt.plot(wvs, rsgspec(wvs), color = 'blue', alpha = 0.5)
plt.xlim(0.3e4, 5e4)
plt.xlabel('Wavelength (Angstrom)')
plt.ylabel(r'photlam $(\text{photons}/\text{s}/\text{cm}^2/\AA)$')

In [ ]:
energy = (const.h*const.c/wvs).to(u.erg)
files = sorted(glob.glob('data/rsg_grf/rsg.s*'))
spec =  np.loadtxt(files[-1], skiprows=4)
dwv, dfl = spec[:, 0], spec[:, 1]
de = spec[:, 4]

dist = 13*u.Mpc
flux_scale = ((1 * u.L_sun) / (4 * np.pi * dist**2)).to(u.erg/u.s/u.cm**2)
scale = 10**4.4

flux = (dfl/dwv)*u.erg/u.s/u.cm**2/u.Angstrom
waves = dwv*u.um
denergy = (const.h*const.c/waves).to(u.erg)

normalize = simpson(flux, x=waves.to(u.Angstrom))
scale_flux = scale * flux_scale * flux.value/normalize / u.Angstrom
dust_flux = de*scale_flux

In [ ]:
wvs = rsgspec.waveset
plt.plot(wvs/1e4, rsgspec(wvs)*energy, color = 'blue', alpha = 0.5, label = 'agraphite')
plt.plot(wvs/1e4, bbspec(wvs)*energy, color = 'orange', alpha = 0.5, label = 'agraphite dust')
plt.plot(waves, scale_flux, color = 'royalblue', label = 'dusty spec')
plt.plot(waves, dust_flux, color = 'coral', label = 'dusty dust')
plt.legend()
plt.xlim(0.3, 12)
plt.xlabel('Wavelength (Angstrom)')
plt.ylabel(r'$f_{\lambda} (\text{erg}/\text{s}/\text{cm}^2/\AA)$')
plt.savefig('plots/dusty_comp_grf.png', bbox_inches='tight')

In [ ]:
wvs = [float(i[1:3])/10 for i in filts]
mags_s2 = [rsgmag_s2[flt].value for flt in filts]
mags_g2 = [rsgmag_g2[flt].value for flt in filts]
plt.plot(wvs, mags_s2, color = 'blue', alpha = 0.5, marker = 'o', linestyle = '--', label = 'silicate dust')
plt.plot(wvs, mags_g2, color = 'orange', alpha = 0.5, marker = 'o', linestyle = '--', label = 'graphite dust')
plt.legend()
plt.grid(alpha = 0.3, linestyle = '--')
plt.xlabel(r'Wavelength (${\mu}m$)')
plt.ylabel('AB mag')
plt.ylim(22, 28.5)
plt.gca().invert_yaxis()

In [ ]:
params = ['lum', 'tau', 'temp']
filts = ['f090w', 'f115w', 'f150w', 'f164n', 'f187n', 'f200w', 'f212n', 'f277w', 'f300m', 'f335m', 'f356w', 'f360m', 'f405n', 'f444w']
modeldf = pd.DataFrame(columns=params+filts)
syn_rsg = rsg_phot()
rsg_temps = np.linspace(3400, 5000, 11) * u.K
taus = np.linspace(0.01, 5, 11)
lums = np.linspace(4, 5.5, 11) * u.Lsun
dust_model = 's2'

if os.path.exists(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv'):
    modeldf = pd.read_csv(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv')
else:
    for lum in tqdm(lums):
        for tau in taus:
            for temp in rsg_temps:
                try:
                    rsgmag = syn_rsg.compute_rsg_mag(filts, tau, lum, temp, 1000 * u.K, 12.474*u.Mpc, dust_model)
                    modeldf.loc[len(modeldf)] = [lum.value, tau, temp.value] + [i.value for i in rsgmag.values()]
                except:
                    modeldf.loc[len(modeldf)] = [lum.value, tau, temp.value] + [np.nan]*len(filts)
    modeldf.to_csv(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv', index=False)

In [ ]:
flt_1, flt_2 = 'f115w', 'f200w'
plt.figure(figsize = (8, 6))
plt.scatter(modeldf[flt_1]-modeldf[flt_2], modeldf[flt_2], marker='o', s=30, c=np.repeat(lums.value, 121),
            cmap='magma', edgecolors='black', lw=1.5, alpha=0.9)
plt.colorbar(pad = 0.02)
plt.xlabel(f'{flt_1.upper()} - {flt_2.upper()}')
plt.ylabel(f'{flt_2.upper()}')
plt.gca().invert_yaxis()
plt.grid(alpha=0.3, linestyle='--')

### GRAMS models

In [ ]:
files = sorted(glob.glob('data/grams_ogrid/*txt'))

In [ ]:
grid = np.loadtxt(files[0], skiprows=15)
wv, flux = grid[:, 0] * u.Angstrom, grid[:, 1] * u.erg/u.cm**2/u.s/u.Angstrom

In [ ]:
fig, ax = plt.subplots(1, 1)
for fl in files:
    grid = np.loadtxt(fl, skiprows=15)
    wv, flux = grid[:, 0] * u.Angstrom, grid[:, 1] * u.erg/u.cm**2/u.s/u.Angstrom
    ax.plot(wv.to(u.um), -2.5*np.log10(flux.value), marker = 's', markersize = 3)
    ax.set_xlim(0.1, 4.4)
    ax.set_ylim(32, 47)
# plt.yscale('log')
plt.gca().invert_yaxis()

### dolphot catalogs

In [ ]:
cat = None
for fl in glob.glob('data/dolphot/ngc5643/proc/ngc5643_0_*.csv'):
    if cat is None:
        cat = pd.read_csv(fl)
    else:
        df_ = pd.read_csv(fl)
        cat = pd.concat([cat, df_])
cat.reset_index(inplace=True)

In [ ]:
def lin(x, m, c):
    return m*x+c

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag']

hist = ax.hist2d(cl, m, bins = 4000, norm = 'log', cmap = 'magma')
ax.scatter(modeldf[f1.lower()] - modeldf[f2.lower()], modeldf[f2.lower()], marker='*', s=300, c=np.repeat(rsg_temps.value, 121),
            cmap='viridis', edgecolors='black', lw=1.5, alpha=0.8, label = 'RSG Model')

ax.axhline(26.19, color = 'red', label='TRGB', linestyle = '-.')
plt.axhline(25, color = 'red', label = 'Luminosity cut', linestyle = '--')
ax.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.22625, 1, 100), lin(np.linspace(0.22625, 1, 100), -8.0, 25+3), linestyle='-.', color = 'orange')

cbar = fig.colorbar(hist[3], ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-2, 2.5)
ax.set_ylim(32.5, 17)
plt.show()

In [ ]:
mask = (m > 17) & (m < 33) & (cl > -4) & (cl < 4)
cmd = np.vstack((cl[mask].to_numpy(), m[mask].to_numpy()))

In [ ]:
from scipy import stats
kernel = stats.gaussian_kde(cmd)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2, f3 = 'F090W', 'F150W', 'F277W'

cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag'] - cat[f3+'_mag']

hist = ax.hist2d(cl, m, bins = 5000, norm = 'log', cmap = 'inferno')
ax.scatter(modeldf[f1.lower()] - modeldf[f3.lower()], modeldf[f2.lower()] - modeldf[f3.lower()], marker='*', s=300, c=np.repeat(rsg_temps.value, 121),
            cmap='viridis', edgecolors='black', lw=1.5, alpha=0.9, label = 'RSG Model')

cbar = fig.colorbar(hist[3], ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2}) - m({f3})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-5, 5)
ax.set_ylim(-5, 5)
plt.show()

In [ ]:
jwst_wv = np.array([0.9, 1.15, 1.5, 1.64, 1.87, 2, 2.12, 2.77, 3, 3.35, 3.56, 3.6, 4.05, 4.44])

In [ ]:
# rsg_mask = (m<24) & (cl < 1.1) & (cl > 0)
rsg_mask = (m<25) & (m > lin(cl, -20.0, 33.0-3)) & (m < lin(cl, -8.0, 25+3.5))
idx = cat[rsg_mask].sample(1).index[0]
st = cat.loc[idx].to_numpy()
mag, magerr = st[7:21], st[21:]
nanmask = (mag>99) | (magerr>9)
mag[nanmask], magerr[nanmask] = np.nan, np.nan
fig, ax = plt.subplots(1, 1)
ax.errorbar(x=jwst_wv, y=mag, yerr=magerr, marker='o', markerfacecolor='white', markeredgecolor='black', 
            markersize=7, lw=0, ecolor='black', elinewidth=1, label = 'dolphot SED')

n_lines = 1331
c = np.arange(1, n_lines + 1)
norm = mpl.colors.Normalize(vmin=c.min(), vmax=c.max())
cmap = mpl.cm.ScalarMappable(norm=norm, cmap=mpl.cm.magma)
cmap.set_array([])

for i, midx in enumerate(modeldf.index):
    model_st = modeldf.iloc[midx][filts].to_numpy().flatten()
    if i==0:
        ax.plot(jwst_wv, model_st, c=cmap.to_rgba(i+1), markeredgecolor = 'black', alpha = 0.2, label = 'MARCS models')
    else:
        ax.plot(jwst_wv, model_st, c=cmap.to_rgba(i+1), markeredgecolor = 'black', alpha = 0.2)

plt.gca().invert_yaxis()
plt.legend()
plt.xlabel(r'Wavelength $(\mu m)$')
plt.ylabel('AB Mag')
# plt.savefig('plots/sed.png', bbox_inches = 'tight')

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag']

hist = ax.hist2d(cl, m, bins = 3000, norm = 'log', cmap = 'inferno')
ax.scatter(cat.loc[idx]['F115W_mag']-cat.loc[idx]['F200W_mag'], cat.loc[idx]['F200W_mag'], marker='*', s=300, facecolor='white', edgecolor='black', alpha=1.0, label = 'RSG Model')

ax.axhline(26.19, color = 'red', label='TRGB', linestyle = '-.')
ax.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.22625, 1, 100), lin(np.linspace(0.22625, 1, 100), -8.0, 25+3.5), linestyle='-.', color = 'orange')

cbar = fig.colorbar(hist[3], ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-4, 4)
ax.set_ylim(32, 18)
plt.axhline(23, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
rsg_mask.sum()

### GP model

In [ ]:
import george
from george import kernels
from sklearn.exceptions import ConvergenceWarning
import warnings

In [ ]:
def fit_gp_model_george(wvs, mags, err, min_len_scale=1):

    # kernel = kernels.Matern52Kernel(metric = [min_len_scale**2], bounds = [[np.min(mags) - 5,np.max(mags)  + 5]])
    kernel = kernels.RationalQuadraticKernel(log_alpha=0.4, metric=min_len_scale) 
    gp_model = george.GP(kernel, solver=george.HODLRSolver, white_noise=np.log(0.01**2), fit_white_noise=True)

    with warnings.catch_warnings():
        warnings.filterwarnings(action="ignore", category=ConvergenceWarning)
        gp_model.compute(wvs, err)
    return gp_model

In [ ]:
def plot_sed_with_gpmodel(wv, mag, magerr, gpmodel, savefilename=None):
    plt.figure()
    plt.errorbar(wv, mag, magerr, fmt='.', markersize=8, color='#0f084b', label = 'dolphot sed')

    x = np.atleast_2d(np.linspace(min(wv), max(wv), 500)).T
    y_pred, sigmasq = gpmodel.predict(mag, x, return_var=True)
    sigma = np.sqrt(sigmasq)

    plt.plot(x, y_pred, c='#26408b', label='GP model')
    plt.fill(np.concatenate([x, x[::-1]]),
             np.concatenate([y_pred - 1.9600 * sigma, (y_pred + 1.9600 * sigma)[::-1]]),
             alpha=.5, fc='#a6cfd5', ec='None', label='95% confidence interval')

    plt.gca().invert_yaxis()
    plt.xlabel(r'Wavelength $(\mu m)$')
    plt.ylabel('AB Mag')
    # plt.ylim(26.5, 24)
    plt.grid(alpha = 0.3, linestyle = '--')
    plt.savefig(savefilename, bbox_inches='tight')
    # plt.show()

In [ ]:
mid, chisq = [], []
nwmask = np.isnan(mag)
for id_ in modeldf.index:
    model_magnitudes = modeldf.loc[id_][['f090w', 'f115w', 'f150w', 'f164n', 'f187n', 'f200w', 'f212n', 'f277w', 'f300m', 'f335m', 'f356w', 'f360m', 'f405n', 'f444w']].to_numpy()
    chsq = np.sum((model_magnitudes[~nwmask] - mag[~nwmask])**2)
    mid.append(id_); chisq.append(chsq)
print(min(chisq))

In [ ]:
gpmag, gpmagerr = mag[~np.isnan(mag)], magerr[~np.isnan(magerr)]
waves = jwst_wv[~np.isnan(mag)]
gpmodel = fit_gp_model_george(waves, gpmag, gpmagerr, min_len_scale=2.5)
plot_sed_with_gpmodel(waves, gpmag, gpmagerr, gpmodel, 'plots/sed_george.png')
best_model = modeldf.loc[mid[np.argmin(chisq)]][['f090w', 'f115w', 'f150w', 'f164n', 'f187n', 'f200w', 'f212n', 'f277w', 'f300m', 'f335m', 'f356w', 'f360m', 'f405n', 'f444w']].to_numpy()
plt.plot(jwst_wv, best_model, marker = 's', fillstyle='none', markeredgecolor = 'royalblue', color = 'cornflowerblue', markersize = 7)
# plt.savefig('plots/marcs_fit7.png', bbox_inches='tight')

In [ ]:
modeldf.loc[mid[np.argmin(chisq)]][['lum', 'tau', 'temp']]

### Base SED fits

In [ ]:
params = ['lum', 'tau', 'temp']
filts = ['f090w', 'f115w', 'f150w', 'f164n', 'f187n', 'f200w', 'f212n', 'f277w', 'f300m', 'f335m', 'f356w', 'f360m', 'f405n', 'f444w']
modeldf = pd.DataFrame(columns=params+filts)
syn_rsg = rsg_phot()
rsg_temps = np.linspace(3400, 5000, 21) * u.K
taus = np.linspace(0.01, 5, 21)
lum = 4.0 * u.Lsun
dust_model = 's2_base'

if os.path.exists(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv'):
    modeldf = pd.read_csv(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv')
else:
    # for lum in tqdm(lums):
    for tau in tqdm(taus):
        for temp in rsg_temps:
            try:
                rsgmag = syn_rsg.compute_rsg_mag(filts, tau, lum, temp, 1000 * u.K, 13.061*u.Mpc, dust_model) #mu = 30.58, Siyang Li+24
                modeldf.loc[len(modeldf)] = [lum.value, tau, temp.value] + [i.value for i in rsgmag.values()]
            except:
                modeldf.loc[len(modeldf)] = [lum.value, tau, temp.value] + [np.nan]*len(filts)
    modeldf.to_csv(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv', index=False)

In [ ]:
modeldf_base = copy.copy(modeldf)
lums = np.array(list(np.linspace(4.05, 5, 20)) + list(np.linspace(5.2, 5.8, 4)))
for lum_ in tqdm(lums):
    del_lum = -2.5 * (lum_ - 4.0)
    bright_df = copy.copy(modeldf_base)
    bright_df[filts] = bright_df[filts] + del_lum
    bright_df['lum'] = lum_
    modeldf = pd.concat([modeldf, bright_df])

modeldf.to_csv(f'data/dolphot/ngc5643/ngc5643_marcs_{dust_model}.csv', index=False)

In [ ]:
# modeldf = pd.read_csv('/Users/aswin/rsg_phot/data/dolphot/ngc5643/ngc5643_marcs_s2_base.csv')
filts = ['f090w', 'f115w', 'f150w', 'f164n', 'f187n', 'f200w', 'f212n', 'f277w', 'f300m', 'f335m', 'f356w', 'f360m', 'f405n', 'f444w']

In [ ]:
wvs = [float(i[1:3])/10 for i in filts]

base = []
for col in modeldf.columns[3:]:
    base.append(max(modeldf[col]))
base = np.array(base)

mmags = modeldf[filts].to_numpy()
for i in mmags:
    plt.plot(wvs, i, color = 'gray', alpha = 0.2)
plt.plot(wvs, base, color = 'royalblue')

plt.gca().invert_yaxis()

In [ ]:
cat = None
for fl in glob.glob('data/dolphot/ngc5643/proc/ngc5643_0_*.csv'):
    if cat is None:
        cat = pd.read_csv(fl)
    else:
        df_ = pd.read_csv(fl)
        cat = pd.concat([cat, df_])
cat.reset_index(inplace=True)

In [ ]:
def calc_chisq(mags, modeldf):
    mask = mags > 10
    flm = np.array(filts)[mask]
    dif = modeldf[flm].to_numpy() - mags[mask]
    chisq = np.sum(dif**2, axis = 1)/mask.sum()
    minchi = np.argmin(chisq)
    return minchi, chisq[minchi]

In [ ]:
def fit_sed(modeldf, cat, filts, base):
    fls = [i.upper()+'_mag' for i in filts]
    cat_mags = cat[fls]
    cat_mags = cat_mags.fillna(value = 99.999)
    cat_mags[cat_mags == 99.999] = 0.5

    mdf = cat_mags-base
    difm = (mdf < 0).all(axis=1)
    ndet = (cat_mags > 15) & (cat_mags < 30) 
    ndetm = ndet.sum(axis=1) > 6
    base_mask = difm & ndetm

    sub_mag_df = cat_mags[base_mask]
    sub_mag_df.loc[:, 'chisq'] = np.nan
    sub_mag_df.loc[:, 'midx'] = np.nan

    for idx in tqdm(sub_mag_df.index):
        mc, cs = calc_chisq(cat_mags[base_mask].loc[idx].to_numpy(), modeldf)
        sub_mag_df.loc[idx, 'chisq'] = cs
        sub_mag_df.loc[idx, 'midx'] = mc

    return sub_mag_df

In [ ]:
def find_rsgs(modeldf, cat, sub_mag_df, chi_cut = 5e-3):
    csq = sub_mag_df['chisq']
    params = modeldf.loc[sub_mag_df[csq < chi_cut]['midx']]
    rsgs = cat.loc[sub_mag_df[csq < chi_cut].index]
    x, y = rsgs['X'], rsgs['Y']
    temp, lum = params['temp'], params['lum']

    return rsgs, x, y, temp, lum

In [ ]:
sub_mag_df = fit_sed(modeldf, cat, filts, base)

In [ ]:
rsgs, x, y, temp, lum = find_rsgs(modeldf, cat, sub_mag_df)

In [ ]:
csq = sub_mag_df['chisq']
_ = plt.hist(np.log10(csq), bins = 40)

In [ ]:
rsgs = cat.loc[sub_mag_df[csq < 5e-3].index]
x, y = rsgs['X'], rsgs['Y']

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (10, 7))
f1, f2 = 'F115W', 'F200W'

cl, m = cat[f1+'_mag'] - cat[f2+'_mag'], cat[f2+'_mag']

hist = ax.hist2d(cl, m, bins = 3000, norm = 'log', cmap = 'inferno')
ax.scatter(rsgs['F115W_mag'] - rsgs['F200W_mag'], rsgs['F200W_mag'], marker='*', s=30, facecolor='white', edgecolor='black', alpha=1.0)

ax.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
ax.plot(np.linspace(0.22625, 1, 100), lin(np.linspace(0.22625, 1, 100), -8.0, 25+3), linestyle='-.', color = 'orange')

cbar = fig.colorbar(hist[3], ax = ax, pad = 0.009)
ax.set_xlabel(rf'$m({f1}) - m({f2})$')
ax.set_ylabel(rf'$m({f2})$')
ax.set_title('NGC5643 CMD')

ax.legend(loc = 'lower right', framealpha = 0)
ax.tick_params(axis='both', which='major')
ax.tick_params(axis='both', which='minor')
cbar.ax.tick_params() 
ax.grid(alpha = 0.3, linestyle = '--')
ax.set_xlim(-4, 4)
ax.set_ylim(32, 18)
plt.axhline(23, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
params = modeldf.loc[sub_mag_df[csq < 5e-3]['midx']]
_ = plt.hist(temp, bins = 35)

In [ ]:
idx = sub_mag_df.sample(1).index[0]
plt.scatter(wvs, sub_mag_df.loc[idx].to_numpy()[:-2])
plt.plot(wvs, modeldf.iloc[int(sub_mag_df.loc[idx, 'midx'])][filts])
print(sub_mag_df.loc[idx, 'chisq'])
plt.ylim(26, 20)

### Clustering

In [ ]:
from sklearn.decomposition import PCA, FastICA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, RationalQuadratic
from sklearn.svm import SVC

In [ ]:
pca = PCA(n_components=6)
ica = FastICA(n_components=5)
std_scaler = StandardScaler()
mmax_scaler = MinMaxScaler()

In [ ]:
fls = [i.upper()+'_mag' for i in filts]
subdf = cat[(m < 26)][fls] # & (cl > -0.5) & (cl < 1.5)
subdf[subdf > 99] = np.nan
subdf = subdf.dropna()

In [ ]:
model_mags = modeldf[filts].to_numpy().T
# stacked_mags = copy.copy(mags)
stacked_mags = np.concat([subdf.values, modeldf[filts].values]).T
icm = np.array(len(subdf)*[True] + 1331*[False])

scaled_mags = mmax_scaler.fit_transform(stacked_mags)
fit_input = stacked_mags
pca.fit(fit_input)

In [ ]:
plt.scatter(pca.components_[0], pca.components_[1], s = 10, color = 'royalblue')
plt.xlabel('PCA_1')
plt.ylabel('PCA_2')
# plt.xlim(-0.002, 0.002)
# plt.ylim(-0.005, 0.005)
plt.grid(linestyle = '--', alpha = 0.5)
plt.show()

In [ ]:
plt.scatter(pca.components_[0][icm], pca.components_[1][icm], s = 10, color = 'royalblue')
plt.scatter(pca.components_[0][~icm], pca.components_[1][~icm], s = 100, marker = '*', color = 'hotpink')
plt.xlabel('PCA_1')
plt.ylabel('PCA_2')
# plt.xlim(-0.002, 0.002)
# plt.ylim(-0.005, 0.005)
plt.grid(linestyle = '--', alpha = 0.5)
plt.show()

In [ ]:
pcacomp0, pcacomp1 = 0, 1
plt.scatter(pca.components_[pcacomp0][icm], pca.components_[pcacomp1][icm], s = 10, color = 'royalblue')
plt.scatter(pca.components_[pcacomp0][~icm], pca.components_[pcacomp1][~icm], s = 100, marker = '*', color = 'hotpink')
plt.xlabel('PCA_1')
plt.ylabel('PCA_2')
# plt.xlim(-0.002, 0.002)
# plt.ylim(-0.005, 0.005)
plt.grid(linestyle = '--', alpha = 0.5)
plt.show()

In [ ]:
model_mags = np.vstack([f090w_mag, f150w_mag, f277w_mag])
smg = []
stacked_mags = np.hstack([mags, model_mags])

mask = (stacked_mags[0] < 80) & (stacked_mags[1] < 80) & (stacked_mags[2] < 80)
smg.append(stacked_mags[0][mask])
smg.append(stacked_mags[1][mask])
smg.append(stacked_mags[2][mask])
smg = np.array(smg)
stacked_mags = smg
icm = np.array(len(mags[0])*[True] + len(model_mags[0])*[False])[mask]

scaled_mags = std_scaler.fit_transform(stacked_mags)
fit_input = stacked_mags
ica.fit(fit_input)

In [ ]:
plt.scatter(ica.components_[0][icm], ica.components_[1][icm], s = 10, color = 'royalblue')
plt.scatter(ica.components_[0][~icm], ica.components_[1][~icm], s = 100, marker = '*', color = 'hotpink')
plt.xlabel('ICA_1')
plt.ylabel('ICA_2')
# plt.xlim(-1e-5, 1e-5)
# plt.ylim(-2e-5, 2e-5)
plt.grid(linestyle = '--', alpha = 0.5)
plt.show()

In [ ]:
plt.scatter(ica.components_[0][icm], ica.components_[1][icm], s = 10, color = 'royalblue')
plt.scatter(ica.components_[0][~icm], ica.components_[1][~icm], s = 100, marker = '*', color = 'hotpink')
plt.xlabel('ICA_1')
plt.ylabel('ICA_2')
# plt.xlim(-1e-5, 1e-5)
# plt.ylim(-2e-5, 2e-5)
plt.grid(linestyle = '--', alpha = 0.5)
plt.show()

In [ ]:
from sklearn.cluster import DBSCAN, HDBSCAN

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(stacked_mags.T)

In [ ]:
np.unique(labels)

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(stacked_mags.T[icm][:, 1][(labels[icm]<10)] - stacked_mags.T[icm][:, 5][(labels[icm]<10)], 
                      stacked_mags.T[icm][:, 5][(labels[icm]<10)], 
                      c=labels[icm][(labels[icm]<10)], cmap = 'tab10', s=5, alpha=0.5)
# plt.scatter(stacked_mags.T[~icm][:, 1] - stacked_mags.T[~icm][:, 5], stacked_mags.T[~icm][:, 5], color='orange', marker = 's', s=2, alpha=0.7)

plt.axhline(26.19, color = 'red', label='TRGB', linestyle = '-.')
plt.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
plt.plot(np.linspace(0.22625, 1, 100), lin(np.linspace(0.22625, 1, 100), -8.0, 25+3), linestyle='-.', color = 'orange')
plt.plot(np.linspace(0.22625, 1.5, 100), lin(np.linspace(0.22625, 1.5, 100), -13.3, 24.6+12), linestyle='-.', color = 'pink')

plt.xlabel('F115W - F200W')
plt.ylabel('F200W')
plt.title('DBSCAN Clustering')
plt.colorbar(scatter, label='Cluster Label')
plt.grid(alpha=0.3, linestyle='--')
plt.xlim(-2, 4)
plt.ylim(26, 15)
plt.show()

In [ ]:
(labels[icm] == 0).sum()

In [ ]:
cat.loc[subdf[labels[icm] == 2].index].to_csv('data/dolphot/ngc5643/dbscan_rsg.csv', index=False)

In [ ]:
fls = [i.upper()+'_mag' for i in filts]
rsg_subdf = cat[(m<25) & (m > lin(cl, -20.0, 33.0-3)) & (m < lin(cl, -8.0, 25+3))][fls]
rsg_subdf[rsg_subdf > 99] = np.nan
rsg_subdf = rsg_subdf.dropna()
rlb = [0]*len(rsg_subdf)

fls = [i.upper()+'_mag' for i in filts]
agb_subdf = cat[(m<26) & (m > lin(cl, -8.0, 25+6)) & (cl < 1.0)][fls]
agb_subdf[agb_subdf > 99] = np.nan
agb_subdf = agb_subdf.dropna()
agb_subdf = agb_subdf.sample(10000)
alb = [1]*len(agb_subdf)

stacked_mags = pd.concat([rsg_subdf, agb_subdf]).values
X, y = stacked_mags, np.array(rlb+alb)

In [ ]:
clf = SVC(probability=True)
clf.fit(X, y)
clf.score(X, y)

In [ ]:
fls = [i.upper()+'_mag' for i in filts]
subdf = cat[(m<26)][fls]
subdf[subdf > 99] = np.nan
subdf = subdf.dropna()
# subdf = subdf.sample(10000)

In [ ]:
probs = clf.predict_proba(subdf.values)
rp, ap = probs[:, 0], probs[:, 1]

In [ ]:
plt.hist(rp, bins = 40)
_ = plt.hist(ap, bins = 40)

In [ ]:
plt.scatter((subdf['F115W_mag'] - subdf['F200W_mag']), subdf['F200W_mag'], s=1.0, c = rp)
plt.colorbar()

# plt.scatter(cl, m, color = 'gray', alpha = 0.2, s = 0.01)
plt.axhline(26.19, color = 'red', label='TRGB', linestyle = '-.')
plt.plot(np.linspace(0.1905, 1, 100), lin(np.linspace(0.1905, 1, 100), -20.0, 33.0-3), linestyle='-.', color = 'coral', label = 'Ren+21')
plt.plot(np.linspace(0.22625, 1, 100), lin(np.linspace(0.22625, 1, 100), -8.0, 25+3), linestyle='-.', color = 'orange')

plt.xlim(-0.5, 1.8)
plt.ylim(26, 18)

plt.grid(alpha=0.3, linestyle='--')

### MIST

In [ ]:
import read_mist_models
from astropy.io import ascii

In [ ]:
eepcmd = read_mist_models.EEPCMD('data/mist/0120000M.track.eep.cmd')

In [ ]:
print('version: ', eepcmd.version)
print('photometric system: ', eepcmd.photo_sys)
print('abundances: ', eepcmd.abun)
print('rotation: ', eepcmd.rot)
print('initial mass: ', eepcmd.minit)
print('Av extinction: ', eepcmd.Av_extinction)
print('available columns: ', eepcmd.hdr_list)


print('initial mass: ', eepcmd.minit)
print('available columns: ', eepcmd.hdr_list)

In [ ]:
eepcmd.plot_CMD(['F115W', 'F200W', 'F200W'], color='Black', phases=[2], 
                phasecolor=['Pink'])

In [ ]:
dat = ascii.read('data/mist/0060000M.track.eep.cmd')
logt, logl = dat['col2'], dat['col4']